# ⚠️ Generation Rules & Databricks Compatibility Reference

This notebook documents the critical rules for generating Databricks-compatible notebooks.
It also serves as a **runnable test** — execute all cells to verify your environment works correctly.

---

## 1. Data Generation Rules

### ✅ CORRECT: Deterministic column-based pseudo-random values
Use `abs(hash(column * prime)) % range + offset` pattern:

In [ ]:
%sql
-- ✅ CORRECT: hash-based deterministic generation
SELECT id,
  abs(hash(id * 7)) % 500 + 1 AS random_value,
  CASE abs(hash(id * 13)) % 4
    WHEN 0 THEN 'Category_A'
    WHEN 1 THEN 'Category_B'
    WHEN 2 THEN 'Category_C'
    ELSE 'Category_D'
  END AS category,
  date_add('2020-01-01', abs(hash(id * 17)) % 1500) AS random_date
FROM (SELECT explode(sequence(1, 10)) AS id);

### ❌ WRONG: These patterns cause errors in Databricks

```sql
-- ERROR: rand() seed must be foldable (constant)
SELECT rand(id) FROM table;           -- FAILS
SELECT rand(id + 1) FROM table;       -- FAILS
SELECT rand(column_expr) FROM table;  -- FAILS

-- WORKS:
SELECT rand() FROM table;     -- non-deterministic
SELECT rand(42) FROM table;   -- constant seed, deterministic
```

In [ ]:
%sql
-- ✅ rand() with constant seed works
SELECT rand(42) AS random_val FROM (SELECT 1);

## 2. LATERAL VIEW Rules

### ✅ CORRECT: Use with table-generating functions only

In [ ]:
%sql
-- ✅ CORRECT: LATERAL VIEW with explode
SELECT id, val
FROM (SELECT 1 AS id, array(10, 20, 30) AS arr)
LATERAL VIEW explode(arr) t AS val;

### ❌ WRONG: Subqueries in LATERAL VIEW

```sql
-- PARSE ERROR: subqueries not allowed in LATERAL VIEW
SELECT * FROM table
LATERAL VIEW (SELECT col FROM other_table) t AS val;
```

## 3. Standard Data Generation Pattern

In [ ]:
%sql
-- The standard pattern for generating test data
SELECT
  id,
  abs(hash(id * 7)) % 100 + 1 AS some_value,
  CASE id % 5
    WHEN 0 THEN 'A' WHEN 1 THEN 'B' WHEN 2 THEN 'C'
    WHEN 3 THEN 'D' ELSE 'E'
  END AS category,
  date_add('2020-01-01', abs(hash(id * 13)) % 1500) AS some_date,
  CONCAT('item_', CAST(id AS STRING)) AS name
FROM (SELECT explode(sequence(1, 20)) AS id);

## 4. input_file_name() Behavior

- Only returns meaningful values when reading from **file-based sources**
- For tables created via INSERT, use a string literal instead

In [ ]:
%sql
-- For non-file tables, use a literal instead of input_file_name()
SELECT 'manual_insert' AS source_file, COUNT(*) AS example
FROM (SELECT explode(sequence(1, 5)) AS id);

## 5. VERSION AS OF (Time Travel)

- Only works after at least one modification (INSERT, UPDATE, DELETE, MERGE)
- Version 0 = initial creation state

In [ ]:
%sql
-- Create a test table for time travel demo
DROP TABLE IF EXISTS time_travel_demo;
CREATE TABLE time_travel_demo (id INT, value STRING);
INSERT INTO time_travel_demo VALUES (1, 'original');

-- Modify it to create version 1
UPDATE time_travel_demo SET value = 'modified' WHERE id = 1;

-- Now we can query version 0
SELECT * FROM time_travel_demo VERSION AS OF 0;

In [ ]:
%sql
-- Current version
SELECT * FROM time_travel_demo;

In [ ]:
%sql
-- Cleanup
DROP TABLE IF EXISTS time_travel_demo;

## 6. GENERATED ALWAYS AS IDENTITY

- Works for surrogate keys
- Cannot INSERT explicit values into IDENTITY columns

In [ ]:
%sql
-- Identity column demo
DROP TABLE IF EXISTS identity_demo;
CREATE TABLE identity_demo (
  sk BIGINT GENERATED ALWAYS AS IDENTITY,
  name STRING
);

INSERT INTO identity_demo (name) VALUES ('Alice'), ('Bob'), ('Charlie');
SELECT * FROM identity_demo;

In [ ]:
%sql
DROP TABLE IF EXISTS identity_demo;

## 7. 2025 Naming Updates

| Current Name | Former Name | Notes |
|---|---|---|
| Lakeflow Spark Declarative Pipelines | Delta Live Tables (DLT) | `import dlt` still works |
| Lakeflow Jobs | Databricks Workflows | API unchanged |
| Declarative Automation Bundles | Databricks Asset Bundles | CLI: `databricks bundle` |
| Unity Catalog Volumes | DBFS | DBFS deprecated |
| Liquid Clustering | Partitioning + ZORDER | For new tables |
| Predictive Optimization | Manual OPTIMIZE/VACUUM | Enabled by default |

## 8. Documentation References

- [Unity Catalog](https://docs.databricks.com/data-governance/unity-catalog/)
- [Auto Loader](https://docs.databricks.com/ingestion/cloud-object-storage/auto-loader/)
- [Liquid Clustering](https://docs.databricks.com/delta/clustering.html)
- [Declarative Pipelines](https://docs.databricks.com/aws/en/ldp/where-is-dlt)
- [Serverless Compute](https://docs.databricks.com/aws/compute/serverless/)
- [DABs](https://docs.databricks.com/aws/dev-tools/bundles/)
- [Delta Sharing](https://docs.databricks.com/aws/delta-sharing/)
- [Predictive Optimization](https://docs.databricks.com/aws/en/optimizations/predictive-optimization)

## 9. Notebook Cell Format Checklist

Every code cell must have:
- `"cell_type": "code"`
- `"metadata": {}`
- `"source": [...]` (list of strings, each ending with `\n` except last)
- `"outputs": []`
- `"execution_count": null`

SQL cells must start with `%sql` on the first line.

Practice cells format:
```
-- ============================================
-- 🎯 YOUR TURN: [Description]
-- ============================================
-- Hint: ...
-- Expected output: ...
--
-- Write your code below:
```

Solution cells format:
```
-- ============================================
-- ✅ SOLUTION
-- ============================================
```

## ✅ Environment Verification Complete

If all cells above executed without errors, your Databricks environment is ready
for the full notebook series.

---
*This notebook is a reference companion. The learning starts in Notebook 01: SQL Foundation.*